# Nettoyage — ruslanmv/ai-medical-chatbot
Pipeline de nettoyage et export au format Alpaca pour fine-tuning LoRA.

**Pré-requis :** avoir exécuté `01_eda.ipynb` et ajusté les seuils si nécessaire.

## 1. Imports

In [1]:
import pandas as pd
import json
import re
from pathlib import Path
from tqdm.notebook import tqdm

OUTPUT_PATH = Path('./medical_dataset_clean.json')

# Seuils de filtrage — ajuster selon les observations de 01_eda.ipynb
PATIENT_MIN = 10
PATIENT_MAX = 1000
DOCTOR_MIN  = 20
DOCTOR_MAX  = 2000

## 2. Chargement

In [2]:
df = pd.read_parquet('hf://datasets/ruslanmv/ai-medical-chatbot/dialogues.parquet')
print(f'Lignes chargées : {len(df):,}')
df.head(2)

Lignes chargées : 256,916


,Description,Patient,Doctor
0,Q. What does abutment of the nerve root mean?,"Hi doctor,I am just wondering what is abutting...",Hi. I have gone through your query with dilige...
1,Q. What should I do to reduce my weight gained...,"Hi doctor, I am a 22-year-old female who was d...",Hi. You have really done well with the hypothy...


## 3. Stats initiales

In [3]:
steps = []

def log_step(label, df_):
    steps.append({'Étape': label, 'Lignes restantes': len(df_), 'Supprimées': steps[-1]['Lignes restantes'] - len(df_) if steps else 0})
    print(f'{label} → {len(df_):,} lignes')

log_step('Chargement initial', df)

Chargement initial → 256,916 lignes


## 4. Suppression des valeurs nulles / vides

In [4]:
df = df.dropna(subset=['Patient', 'Doctor'])
df = df[df['Patient'].str.strip().ne('') & df['Doctor'].str.strip().ne('')]
log_step('Suppression nulls/vides', df)

Suppression nulls/vides → 256,916 lignes


## 5. Déduplication

In [5]:
df = df.drop_duplicates(subset=['Patient', 'Doctor'])
log_step('Déduplication (Patient + Doctor)', df)

Déduplication (Patient + Doctor) → 246,527 lignes


## 6. Nettoyage texte

In [6]:
def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)  # chars de contrôle
    text = re.sub(r'[ \t]+', ' ', text)                              # espaces multiples
    text = re.sub(r'\n{3,}', '\n\n', text)                          # sauts de ligne excessifs
    return text.strip()

df = df.copy()
df['Patient'] = df['Patient'].map(clean_text)
df['Doctor']  = df['Doctor'].map(clean_text)
log_step('Nettoyage texte', df)

Nettoyage texte → 246,527 lignes


## 7. Filtrage par longueur

In [7]:
patient_len = df['Patient'].str.len()
doctor_len  = df['Doctor'].str.len()

mask = (
    patient_len.between(PATIENT_MIN, PATIENT_MAX) &
    doctor_len.between(DOCTOR_MIN,  DOCTOR_MAX)
)
df = df[mask]
log_step(f'Filtrage longueur (P:{PATIENT_MIN}-{PATIENT_MAX} / D:{DOCTOR_MIN}-{DOCTOR_MAX})', df)

Filtrage longueur (P:10-1000 / D:20-2000) → 236,323 lignes


## 8. Formatage Alpaca (instruction / input / output)

In [9]:
INSTRUCTION = (
    "You are a medical doctor. Answer the following patient question clearly, "
    "professionally and concisely. Do not provide a diagnosis; recommend consulting "
    "a healthcare professional for serious concerns."
)

records = [
    {
        'instruction': INSTRUCTION,
        'input': row['Patient'],
        'output': row['Doctor']
    }
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Formatage')
]

print(f'\nExemple de record :')
print(json.dumps(records[0], indent=2, ensure_ascii=False))

Formatage:   0%|          | 0/236323 [00:00<?, ?it/s]


Exemple de record :
{
  "instruction": "You are a medical doctor. Answer the following patient question clearly, professionally and concisely. Do not provide a diagnosis; recommend consulting a healthcare professional for serious concerns.",
  "input": "Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for annular bulging and tear?",
  "output": "Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->"
}


## 9. Export JSON

In [10]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

size_mb = OUTPUT_PATH.stat().st_size / 1e6
print(f'Fichier exporté : {OUTPUT_PATH}')
print(f'Taille : {size_mb:.1f} MB  |  {len(records):,} entrées')

Fichier exporté : medical_dataset_clean.json
Taille : 279.7 MB  |  236,323 entrées


## 10. Rapport final

In [ ]:
report = pd.DataFrame(steps)
initial = report.iloc[0]['Lignes restantes']
final   = report.iloc[-1]['Lignes restantes']
retention = final / initial * 100

print('=== Rapport de nettoyage ===')
display(report)
print(f'\nTaux de rétention : {retention:.1f}%  ({final:,} / {initial:,} lignes)')
print(f'Fichier prêt pour fine-tuning : {OUTPUT_PATH.resolve()}')

=== Rapport de nettoyage ===


,Étape,Lignes restantes,Supprimées
0,Chargement initial,256916,0
1,Suppression nulls/vides,256916,0
2,Déduplication (Patient + Doctor),246527,10389
3,Nettoyage texte,246527,0
4,Filtrage longueur (P:10-1000 / D:20-2000),236323,10204



Taux de rétention : 92.0%  (236,323 / 256,916 lignes)
Fichier prêt pour fine-tuning : C:\Users\noecr\Documents\Ynov\Hackathon_Ynov_IA-M1\datasets\medical_dataset_clean.json
